# Vault workflow — designating, validating, and uploading vacua

This tutorial walks through the **end-to-end vault flow** for a
StringForge user: produce vacua with `jaxvacua`, store them in a local
vault, validate them via `stringforge.vacuavault`, and upload the
validated subset to the community HuggingFace dataset
`aschachner/vacua_vault`.

The validation layer lives in **`stringforge.vacuavault`** so it's
shared across all downstream packages (jaxvacua, kahlerjax, …).
The validator uses **pure dependency injection**: no
jaxvacua imports inside `stringforge.vacuavault`; the caller passes
in the database and the model-hash function.

| Step | Package owns it | What you do |
| --- | --- | --- |
| Produce vacua | `jaxvacua` | `promote(...)` / `enumerate(...)` etc. |
| Store locally | `jaxvacua.vacua_writer` (uses `stringforge.cy_io`) | `db.designate_vacua(...)` |
| **Validate** | **`stringforge.vacuavault`** | `vv.validate_parquet_file(...)` |
| Upload to HF | `jaxvacua.vacua_writer` | `db.push_vacua_to_hub(...)` |
| Fetch curated | `jaxvacua.vacua_writer` | `db.fetch_vacua_from_hub(...)` |


## Step 0 — Setup

In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# StringForge provides the validation layer.
import stringforge as sj
from stringforge import vacuavault as vv

# Sandbox the vault for this demo so we don't pollute the real one.
tmpdir = tempfile.mkdtemp(prefix="vault_demo_")
sj.set_vault_dir(os.path.join(tmpdir, "vault"))
sj.set_data_dir(os.path.join(tmpdir, "cache"))

print("stringforge version :", sj.__version__)
print("vault dir         :", os.environ.get("STRINGFORGE_VAULT"))
print("public surface    :", sorted(vv.__all__)[:6], "...")


(CVXPY) Apr 30 05:23:16 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Apr 30 05:23:16 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


stringjax version : 0.0.2
vault dir         : /var/folders/p7/f3t072gd3rbfgb00y3hj7dwc0000gn/T/vault_demo_ww9o338k/vault
public surface    : ['LABEL_SLUG_RE', 'RESERVED_NAMES', 'SCHEMA_VERSION', '_check_identity_consistency', '_extract_identity_from_row', '_identity_from_row'] ...


/Users/andreasschachner/Software/GitHub/jax-work/kahlerjax/kahlerjax/cytree.py:46: UserWarning: TODO: Two neighbors is broken!
  warnings.warn("TODO: Two neighbors is broken!")


## Step 1 — A minimal synthetic vacua parquet

A real workflow would obtain the dataframe from
`db.designate_vacua(...)` (see Step 5 below).  For the demo we
hand-build a tiny parquet file with the **required columns** and
**identity columns** so the validator has something to chew on.

Required columns (from `vv.REQUIRED_COLUMNS`):
* `flux`        — integer vector of fluxes
* `moduli_re`, `moduli_im` — complex-structure moduli
* `tau_re`, `tau_im`       — axio-dilaton

Identity columns (used by the homogeneity check + auto-load path):
* `ks_id`, `triang_id`, `cicy_id`, `h11`, `h12`, `model_hash`


In [2]:
def make_demo_df(n=5, ks_id=29, triang_id=1, h12=2):
    rng = np.random.default_rng(42)
    n_fluxes = 2 * (h12 + 1)
    return pd.DataFrame({
        "flux":      [[int(x) for x in rng.integers(-3, 4, 2 * n_fluxes)] for _ in range(n)],
        "moduli_re": [[1.0, 2.0]] * n,
        "moduli_im": [[3.0, 4.0]] * n,
        "tau_re":    [0.0] * n,
        "tau_im":    [1.5] * n,
        "is_susy":   [True] * n,
        # Identity columns:
        "ks_id":     [ks_id]    * n,
        "triang_id": [triang_id] * n,
        "cicy_id":   [-1]       * n,
        "h11":       [h12]      * n,
        "h12":       [h12]      * n,
    })


def write_parquet(df, path, metadata):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    md_existing = dict(table.schema.metadata or {})
    md_existing.update({k.encode(): str(v).encode() for k, v in metadata.items()})
    table = table.replace_schema_metadata(md_existing)
    pq.write_table(table, path)


df = make_demo_df()
parquet_path = os.path.join(
    tmpdir, "vault", "tdf", "h12_2", "ks_29_tri_1", "SUSY_Nmax34.parquet")
write_parquet(df, parquet_path, {
    "schema_version": vv.SCHEMA_VERSION,
    "susy":           "SUSY",
    "method":         "enumerate",
    "nmax":           34,
})
print(f"wrote {len(df)} rows to:")
print(f"  {parquet_path}")


wrote 5 rows to:
  /var/folders/p7/f3t072gd3rbfgb00y3hj7dwc0000gn/T/vault_demo_ww9o338k/vault/tdf/h12_2/ks_29_tri_1/SUSY_Nmax34.parquet


## Step 2 — Schema-only validation

The cheapest mode: no model, no physics.  Catches reserved
filenames, missing columns, schema-version mismatches, and
non-integer flux entries.  Set `physics_checks="off"` to skip
**both** the identity-homogeneity check and any auto-load attempt.


In [3]:
result = vv.validate_parquet_file(parquet_path, physics_checks="off")

print(f"passed   : {result['passed']}")
print(f"n_rows   : {result['n_rows']}")
print(f"n_failed : {result['n_failed']}")
print(f"errors   : {result['errors']}")
print(f"warnings : {result['warnings']}")


passed   : True
n_rows   : 5
n_failed : 0
errors   : []
warnings : []


## Step 3 — Identity-homogeneity check

`physics_checks="auto"` (the default) runs the schema checks
**plus** an identity-consistency check — every row in a parquet
must describe the same model (one-file-per-model is a hard
invariant of the vault layout).  Mixed-model files are rejected.


In [4]:
# Synthesise a *bad* parquet with rows split across two ks_ids.
df_a = make_demo_df(n=2, ks_id=29, triang_id=1)
df_b = make_demo_df(n=2, ks_id=42, triang_id=0)
df_mixed = pd.concat([df_a, df_b], ignore_index=True)

bad_path = os.path.join(tmpdir, "bad_mixed.parquet")
write_parquet(df_mixed, bad_path, {"schema_version": vv.SCHEMA_VERSION})

result = vv.validate_parquet_file(bad_path, physics_checks="auto")
print(f"passed   : {result['passed']}")
print(f"errors   : {result['errors']}")


passed   : False
errors   : ['file contains rows for 2 distinct models (violates the one-model-per-file invariant). Identifier tuples found: [[29, 1, -1], [42, 0, -1]]']


## Step 4 — Physics-aware validation (three modes)

Three modes for `physics_checks`:

| Mode | Behaviour |
|---|---|
| `"off"` | Schema-only.  No identity check, no physics. |
| `"auto"` (default) | Schema + identity homogeneity always.  Physics if the row identity + a `db=` are available; else **warning**. |
| `"explicit"` | Same as `"auto"` but missing `db=` / load failure → top-level **error**. |

The deeper validation reconstructs the model from row identity
and runs tadpole + F-term checks.  Because `stringforge.vacuavault`
is jaxvacua-free by design, **you pass the database and the
model-hash function in yourself** — pure dependency injection.

In `"auto"` mode without a `db=`, the validator is *graceful*:
it logs a warning and skips physics rather than failing.  Watch:


In [5]:
# `auto` mode without a db — warns, doesn't fail.
result = vv.validate_parquet_file(parquet_path, physics_checks="auto")
print(f"passed   : {result['passed']}")
print(f"warnings : {result['warnings']}")


passed   : True
warnings : ["auto-load requires `db=`; pass an LCSDatabase (or any CYDatabase subclass with a `load_model(**ident)` method) instance.  Pass `physics_checks='off'` to skip physics validation."]


In [6]:
# `explicit` mode without a db — escalates the same condition to
# a top-level error.  Useful in CI when you *require* physics
# validation.
result = vv.validate_parquet_file(parquet_path, physics_checks="explicit")
print(f"passed   : {result['passed']}")
print(f"errors   : {result['errors']}")


passed   : False
errors   : ["auto-load requires `db=`; pass an LCSDatabase (or any CYDatabase subclass with a `load_model(**ident)` method) instance.  Pass `physics_checks='off'` to skip physics validation."]


For real physics validation, plug in jaxvacua's database and
hash function.  We don't run this here (it requires a live
HuggingFace cache), but the call is just:

```python
from jaxvacua.lcs_database import LCSDatabase
from jaxvacua.vacua_writer import _compute_model_hash

db = LCSDatabase(dataset="tdf")
result = vv.validate_parquet_file(
    parquet_path,
    db=db,
    model_hash_fn=_compute_model_hash,
    physics_checks="explicit",   # require physics to pass
)
```

That's the **whole jaxvacua coupling** — two imports, two kwargs.


## Step 5 — End-to-end pipeline (designate + push to HF)

Now you've seen validation in isolation.  In practice it sits
between two `vacua_writer` operations: **designate** to the local
vault, then **push** to the HF community repo.  The push step
calls `_validate_for_upload` internally — your prior call to
`vv.validate_parquet_file` is a *belt-and-braces* check you can
run before invoking the upload.

```python
from jaxvacua.lcs_database import LCSDatabase

db = LCSDatabase(dataset="tdf")
model = db.load_model(ks_id=29, triang_id=1)

# 1. Produce vacua with jaxvacua's solvers (not shown here)
vacua_df = ...

# 2. Designate to the local vault — writes to STRINGFORGE_VAULT
db.designate_vacua(
    vacua_df, label="SUSY_Nmax34",
    committed_by="andreas@example.com",
    model=model,
)

# 3. Validate the freshly-designated parquet (this notebook's topic)
parquet_path = db._resolve_vacua_dir(model=model) / "SUSY_Nmax34.parquet"
report = vv.validate_parquet_file(
    parquet_path,
    db=db,
    model_hash_fn=_compute_model_hash,
    physics_checks="explicit",   # error out on any degradation
)
assert report["passed"], report["errors"]

# 4. Push to HF — opens a PR against aschachner/vacua_vault
db.push_vacua_to_hub(
    vacua_df, label="SUSY_Nmax34",
    committed_by="andreas@example.com",
    model=model,
    create_pr=True,
)
```

Each step has a clear owner — `jaxvacua.vacua_writer` for
designate/push, `stringforge.vacuavault` for validate.  The two
talk through the parquet file on disk and through the
`db=`+`model_hash_fn=` injection.


## Step 6 — `split_by_validation` for partial uploads

If only some rows fail validation, `split_by_validation` returns
two DataFrames so you can upload the valid ones and stash the
rest in a `_rejected/` namespace:


In [7]:
result = vv.validate_parquet_file(parquet_path, physics_checks="off")
df_loaded = pd.read_parquet(parquet_path)

valid, rejected = vv.split_by_validation(df_loaded, result["per_row_report"])
print(f"valid    : {len(valid)} rows")
print(f"rejected : {len(rejected)} rows  (columns include 'error': "
      f"{'error' in rejected.columns})")


valid    : 5 rows
rejected : 0 rows  (columns include 'error': False)


## Step 7 — Server-side CLI (HF dataset repo)

When the HF dataset repo's CI runs, it doesn't need to install
jaxvacua.  It invokes `stringforge.vacuavault` directly:

```bash
python -m stringforge.vacuavault validate \
    --repo-path . \
    --base-branch main \
    --report-path validation_report.json \
    --preview-path catalog_preview.md

python -m stringforge.vacuavault rebuild_catalog --repo-path .
python -m stringforge.vacuavault curate community/alice_dS_v2.parquet
```

The CI flow is **schema-only** by design (lightweight, no JAX).
Physics-aware checks happen at user-side validation (Steps 2-5
above).


## Summary

| Where | Module | What it does |
| --- | --- | --- |
| **stringforge** | `stringforge.cy_io` | Generic CY-database I/O.  Reads `cy-database`. |
| **stringforge** | `stringforge.vacuavault` | Schema, validators, catalog tools.  No jaxvacua imports. |
| **jaxvacua** | `jaxvacua.lcs_database` | `LCSDatabase` (subclass of `CYDatabase`); model construction. |
| **jaxvacua** | `jaxvacua.vacua_writer` | `VacuaWriter`: designate / push / fetch.  Hash + identity. |

**The validation layer (`stringforge.vacuavault`) is genuinely
domain-free**: it speaks parquet + pyarrow + the schema rules.
Anything physics-flavoured (`db`, `model`, `model_hash_fn`) gets
injected by the caller.  Sibling packages (kahlerjax, jaxiverse)
can use the same vault layer the same way.


In [8]:
# Cleanup the demo sandbox.
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
sj.set_vault_dir(None)   # clears STRINGFORGE_VAULT env override
